# A comparison of holography X-ray phase contrast imaging:

# mean intensity vs intensity correlations

X-ray holography with intensity correlations is given by the following forward problem:
$$F(f)=c_{f} ~~\text{with}~~ c_{f}(x,y):=\operatorname{Cov}(I(x),I(y)),~~\text{and}~~I:=|\mathcal{D}e^{f}u|^2 $$
The forward operator is given by:
$$
F(f):=|\mathcal{D}M_{\exp(f)}\operatorname{Cov}[u]M_{\exp(f)}^*\mathcal{D}^*|^2.
$$
In contrast to the imaging from intensity correlations, the X-ray holography with mean intensity is given by the following forward problem:
$$G(f)=d_{f} ~~\text{with}~~d_{f}(x):=c_{f}(x,x)=\mathbb{E}[I(x)]^2$$
The forward operator is given by:
$$
G(f):=\operatorname{Diag}(\mathcal{D}M_{\exp(f)}\operatorname{Cov}[u]M_{\exp(f)}^*\mathcal{D}^*).
$$

In [ ]:
import os
import sys

#sys.path.append(os.path.join(os.path.dirname(__file__), '../../'))

from regpy.hilbert import L2
from regpy.vecsps import UniformGridFcts
from regpy.vecsps import NumPyVectorSpace
from regpy.solvers import Setting
from regpy.solvers.nonlinear.fista import FISTA
import regpy.stoprules as rules
from regpy.operators import FresnelPropagator, SquaredModulus, FourierInterpolationOperator
from regpy.functionals import QuadraticNonneg, QuadraticBilateralConstraints
import matplotlib.pyplot as plt

from auxiliary_ops import ReIm, fresnel_prop
from phaseless_passive_ip_ops import Contrast2FactorPhasedCovOp,  Tau, MatrixAutoProductOp, ExpectationCoxModGaussian, CovarianceCoxModGaussian
from create_Vcov import _create_Vcov
from graphic_utils import test_image_cells, test_image_circle_cross, show_measurements, show_results, show_comparison_results, show_measurements

import numpy as np
from numpy.linalg import norm
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)


# Create forward operators and synthetic data

Setting up parameters 

In [ ]:
N=256                             # Pixel number in each spatial direction
#contrast = test_image_circle_cross(N,N)
contrast = test_image_cells(circle_strength=0)
N_frame=9                         # Number of frames ( or realizations)
T=1e6                             # the observation time or the number of photon counts
#fresnel_number=10/(2*np.pi)       # Fresnel number
fresnel_number = 100
pad_amount = None
Fourier_truncation_amount = None
N_b=4                             # denotes the rank of the matrix V

Diminish contrast size keeping pixel number fixed to increase spatial coherence relative to the size of the contrast 

construct decomposition $\operatorname{Cov}[u]=VV^*$ of the source covariance operator 

In [ ]:
sigma=0.5                         # parameter used in the rapid deacaying function
grid = UniformGridFcts((-1,1,N),(-1,1,N),dtype=complex,periodic=True)
Vcov, S=_create_Vcov(N, N_b, grid=grid, sigma=sigma)
# here S is the singular values of the matrix V
#Vcov = Vcov * (1./S)[(None,None,...)]

compute forward operators 

In [ ]:
fp = FresnelPropagator(grid,fresnel_number,pad_amount=pad_amount,Fourier_truncation_amount=Fourier_truncation_amount)
Mat_op=Contrast2FactorPhasedCovOp(Vcov,fp)
ReIm_op=ReIm(grid)

# forward operator associated with the intensity correlations
Cov_Cox = CovarianceCoxModGaussian(Mat_op.codomain,fp.codomain,T)
op_intcorr = Cov_Cox * Mat_op * ReIm_op.adjoint

# forward operator associated with the mean intensity
E_Cox = ExpectationCoxModGaussian(Mat_op.codomain,fp.codomain)
op_meanint = E_Cox * Mat_op * ReIm_op.adjoint

In [ ]:

im = fp(np.exp(contrast)*Vcov[:,:,0])
fp3 = FresnelPropagator(grid,fresnel_number,pad_amount=100,Fourier_truncation_amount=None)
im3 = fp3(np.exp(contrast)*Vcov[:,:,0])

fig, (ax1,ax2) = plt.subplots(1,2)
ax1.imshow(np.abs(im))
ax1.set_title('without zero padding')
ax2.imshow(np.abs(im3))
ax2.set_title('with zero padding')

from numpy.linalg import norm
norm(np.exp(contrast)*Vcov[:,:,0]), norm(im), norm(im3)


Create synthetic intensity data 

In [ ]:
ptw_detection= SquaredModulus(fp.codomain)
taumat_f=Mat_op(contrast) 
taumat_0=Mat_op(np.zeros_like(contrast)) 
data_meanint=ptw_detection.codomain.zeros()   # mean intensity 
intensities=np.zeros((N_frame,)+fp.codomain.shape)
intensities0=np.zeros((N_frame,)+fp.codomain.shape)
uinc=np.zeros((N_frame,)+fp.codomain.shape,dtype=complex)

for i in range(0, N_frame):
    random=1/np.sqrt(2)*(np.random.randn(N_b)+complex(0,1)*np.random.randn(N_b))
    uf=np.tensordot(taumat_f, random, axes=([-1], [0]))
    signal=ptw_detection(uf)
    u0=np.tensordot(taumat_0, random, axes=([-1], [0]))
    signal0=ptw_detection(u0)
    

    #Cox-process
    signal=(1/T)*np.random.poisson(lam=T*signal)
    intensities[i, :, :]=signal
    intensities0[i, :, :]=signal0   
    uinc[i,:,:] = Vcov @ random

In [ ]:
def show_measurements(intensities,vmin=0,vmax=None):
     #cmap='Reds'
     fontsize=10
     levels=40

     row=3
     col=2
     fig, axs = plt.subplots(row, col, figsize=(12,12*row/col))

     for i in range(row):
         for j in range(col):
             axs[i,j].imshow(intensities[col*i+j,:,:], vmax=vmax)
             axs[i,j].axis('off')



In [ ]:
show_measurements(intensities)

In [ ]:
show_measurements(uinc.real,vmin=-0.02,vmax=0.02)

In [ ]:
show_measurements(intensities0)